# 📊 NumPy and Pandas Mastery for ML

> **Master data manipulation and numerical computing for machine learning**

This notebook provides comprehensive coverage of NumPy and Pandas - the foundation libraries for data science and machine learning in Python.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Master** NumPy array operations and broadcasting
- **Excel** at Pandas data manipulation and analysis
- **Understand** efficient data processing techniques
- **Practice** real-world data cleaning scenarios
- **Build** robust data preprocessing pipelines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All imports successful!")

## 🔢 Advanced NumPy Operations

In [ ]:
# Advanced array creation and manipulation
data = np.random.randn(1000, 5)
print(f"Data shape: {data.shape}")

# Statistical operations
print(f"Mean: {data.mean(axis=0)}")
print(f"Std: {data.std(axis=0)}")
print(f"Correlation matrix shape: {np.corrcoef(data.T).shape}")

# Boolean indexing and filtering
outliers = data[np.abs(data) > 2]
print(f"Number of outliers: {len(outliers)}")

# Advanced indexing
indices = np.where(data[:, 0] > 1)
filtered_data = data[indices]
print(f"Filtered data shape: {filtered_data.shape}")

In [ ]:
# Broadcasting examples
X = np.random.randn(100, 3)
mean = X.mean(axis=0)  # Shape: (3,)
std = X.std(axis=0)    # Shape: (3,)

# Broadcasting in standardization
X_standardized = (X - mean) / std  # Broadcasting (100,3) - (3,) / (3,)

print(f"Original mean: {X.mean(axis=0)}")
print(f"Standardized mean: {X_standardized.mean(axis=0)}")
print(f"Standardized std: {X_standardized.std(axis=0)}")

## 📊 Pandas Data Manipulation Mastery

In [ ]:
# Create a realistic dataset
np.random.seed(42)
n_samples = 1000

# Generate synthetic customer data
data = {
    'customer_id': range(1, n_samples + 1),
    'age': np.random.normal(35, 12, n_samples).astype(int),
    'income': np.random.lognormal(10.5, 0.5, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples, p=[0.3, 0.4, 0.25, 0.05]),
    'experience': np.random.exponential(5, n_samples),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], n_samples),
    'purchase_amount': np.random.gamma(2, 50, n_samples),
    'signup_date': pd.date_range('2020-01-01', periods=n_samples, freq='D')[:n_samples]
}

# Introduce some missing values
missing_indices = np.random.choice(n_samples, size=50, replace=False)
data['income'][missing_indices[:25]] = np.nan
data['experience'][missing_indices[25:]] = np.nan

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Data exploration and info
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSummary statistics:")
df.describe()

In [ ]:
# Advanced data manipulation

# 1. Feature engineering
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 100], labels=['Young', 'Adult', 'Middle', 'Senior'])
df['income_per_year_exp'] = df['income'] / (df['experience'] + 1)
df['days_since_signup'] = (datetime.now() - df['signup_date']).dt.days

# 2. Groupby operations
city_stats = df.groupby('city').agg({
    'age': ['mean', 'std'],
    'income': ['mean', 'median'],
    'purchase_amount': ['sum', 'count']
}).round(2)

print("City statistics:")
print(city_stats)

# 3. Pivot tables
pivot = df.pivot_table(
    values='purchase_amount',
    index='education',
    columns='age_group',
    aggfunc='mean'
).round(2)

print("\nPurchase amount by education and age group:")
print(pivot)

## 🧹 Data Cleaning Pipeline

In [ ]:
class DataCleaningPipeline:
    def __init__(self):
        self.numeric_imputers = {}
        self.categorical_imputers = {}
        self.scalers = {}
    
    def handle_missing_values(self, df, strategy='median'):
        """Handle missing values in the dataset"""
        df_clean = df.copy()
        
        # Numeric columns
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df_clean[col].isnull().any():
                if strategy == 'median':
                    fill_value = df_clean[col].median()
                elif strategy == 'mean':
                    fill_value = df_clean[col].mean()
                else:
                    fill_value = 0
                
                df_clean[col].fillna(fill_value, inplace=True)
                self.numeric_imputers[col] = fill_value
        
        # Categorical columns
        categorical_cols = df_clean.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df_clean[col].isnull().any():
                mode_value = df_clean[col].mode()[0]
                df_clean[col].fillna(mode_value, inplace=True)
                self.categorical_imputers[col] = mode_value
        
        return df_clean
    
    def detect_outliers(self, df, columns=None, method='iqr'):
        """Detect outliers using IQR or Z-score method"""
        if columns is None:
            columns = df.select_dtypes(include=[np.number]).columns
        
        outlier_indices = set()
        
        for col in columns:
            if method == 'iqr':
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index
            
            elif method == 'zscore':
                z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
                outliers = df[z_scores > 3].index
            
            outlier_indices.update(outliers)
        
        return list(outlier_indices)
    
    def encode_categorical(self, df, method='onehot'):
        """Encode categorical variables"""
        df_encoded = df.copy()
        categorical_cols = df_encoded.select_dtypes(include=['object', 'category']).columns
        
        for col in categorical_cols:
            if method == 'onehot':
                dummies = pd.get_dummies(df_encoded[col], prefix=col)
                df_encoded = pd.concat([df_encoded, dummies], axis=1)
                df_encoded.drop(col, axis=1, inplace=True)
            
            elif method == 'label':
                df_encoded[col] = pd.Categorical(df_encoded[col]).codes
        
        return df_encoded
    
    def fit_transform(self, df):
        """Complete data cleaning pipeline"""
        print("🧹 Starting data cleaning pipeline...")
        
        # 1. Handle missing values
        df_clean = self.handle_missing_values(df)
        print(f"✅ Missing values handled")
        
        # 2. Detect outliers
        outlier_indices = self.detect_outliers(df_clean)
        print(f"⚠️  Detected {len(outlier_indices)} outliers")
        
        # 3. Encode categorical variables
        df_encoded = self.encode_categorical(df_clean)
        print(f"✅ Categorical variables encoded")
        
        print(f"🎉 Pipeline complete! Shape: {df_encoded.shape}")
        
        return df_encoded, outlier_indices

# Test the pipeline
pipeline = DataCleaningPipeline()
df_processed, outliers = pipeline.fit_transform(df)

print(f"\nOriginal shape: {df.shape}")
print(f"Processed shape: {df_processed.shape}")
print(f"Number of outliers: {len(outliers)}")

## 🎯 Practice Problems

### **Problem 1: Advanced Groupby Operations**
Create complex aggregations and transformations.

In [ ]:
def advanced_customer_analysis(df):
    """
    Perform advanced customer analysis
    
    Tasks:
    1. Calculate customer lifetime value by city
    2. Find top 10% customers by purchase amount
    3. Create customer segments based on behavior
    4. Calculate rolling averages
    
    Returns:
    dict: Analysis results
    """
    # Your code here
    pass

# Test your function
# results = advanced_customer_analysis(df)

### **Problem 2: Time Series Operations**
Work with datetime data and time-based features.

In [ ]:
def create_time_features(df, date_column):
    """
    Create time-based features from datetime column
    
    Features to create:
    - Day of week
    - Month
    - Quarter
    - Is weekend
    - Days since epoch
    
    Returns:
    pd.DataFrame: DataFrame with new time features
    """
    # Your code here
    pass

# Test your function
# df_with_time_features = create_time_features(df, 'signup_date')

## 🎯 Key Takeaways

1. **NumPy broadcasting** enables efficient vectorized operations
2. **Pandas groupby** is powerful for aggregations and transformations
3. **Data cleaning pipelines** ensure reproducible preprocessing
4. **Missing value handling** requires domain knowledge
5. **Feature engineering** can significantly improve model performance

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Experiment with real datasets** from Kaggle
3. **Move to the next notebook**: Data Visualization

---

**Excellent progress!** 🎉 You now have solid data manipulation skills.